# NB05 · Semana 10 — Primer DAG en Airflow

**Ingeniería de Datos — LCD · UCA Rosario · 2026**  
Versión: **para el alumno (a completar)**

## Objetivo

Llevar el pipeline (extracción → bronze → dbt → gold) a un **DAG de Airflow**: definir tareas, dependencias, *scheduling* y reintentos. Este notebook no ejecuta Airflow (eso ocurre en el contenedor `airflow-scheduler`): sirve para **construir y validar** el archivo del DAG antes de desplegarlo.

> Base de la **Entrega 3 (E3)**.

## 1. Anatomía de un DAG

Un DAG (grafo acíclico dirigido) declara tareas y el orden entre ellas. Escribimos el DAG a la carpeta `dags/`, que el scheduler de Airflow lee automáticamente.

## 2. Construir el DAG del proyecto

Completá la definición de tareas y sus dependencias. El DAG de referencia completo está en `dags/pipeline_bcra.py`.

In [25]:

dag_code = r'''
from __future__ import annotations

import pendulum
from airflow.models.dag import DAG
from airflow.operators.bash import BashOperator

default_args = {
    "owner": "grupo03",
    "retries": 0,
}

with DAG(
    dag_id="pipeline_grupo03",
    start_date=pendulum.datetime(
        2024, 1, 1,
        tz="America/Argentina/Cordoba",
    ),
    schedule=None,
    catchup=False,
    max_active_runs=1,
    default_args=default_args,
    tags=["proyecto-integrador", "vatsim", "grupo03"],
) as dag:

    extraer = BashOperator(
        task_id="extraer",
        bash_command=r"""
echo "=== DIAGNOSTICO DBT Y MINIO GRUPO03 ==="

echo ""
echo "1. Comprobando dbt:"
if command -v dbt >/dev/null 2>&1; then
    dbt --version
else
    echo "DBT NO ENCONTRADO"
fi

echo ""
echo "2. Comprobando librerias de Python:"
python - <<'PY'
import importlib.util

for paquete in ["duckdb", "dbt", "deltalake", "boto3"]:
    estado = (
        "DISPONIBLE"
        if importlib.util.find_spec(paquete)
        else "NO DISPONIBLE"
    )
    print(f"{paquete}: {estado}")
PY

echo ""
echo "3. Variables de entorno:"
for variable in \
    DBT_DUCKDB_PATH \
    MINIO_BUCKET \
    MINIO_ENDPOINT \
    MINIO_ACCESS_KEY \
    MINIO_SECRET_KEY \
    AWS_ACCESS_KEY_ID \
    AWS_SECRET_ACCESS_KEY \
    AWS_ENDPOINT_URL
do
    if [ -n "${!variable:-}" ]; then
        echo "$variable: CONFIGURADA"
    else
        echo "$variable: NO CONFIGURADA"
    fi
done

echo ""
echo "4. Proyecto dbt:"
ls -ld /opt/airflow/dags/grupo03/dbt
ls -l /opt/airflow/dags/grupo03/dbt/dbt_project.yml

echo ""
echo "DIAGNOSTICO FINALIZADO"
""",
    )
'''

print(dag_code)





from __future__ import annotations

import pendulum
from airflow.models.dag import DAG
from airflow.operators.bash import BashOperator

default_args = {
    "owner": "grupo03",
    "retries": 0,
}

with DAG(
    dag_id="pipeline_grupo03",
    start_date=pendulum.datetime(
        2024, 1, 1,
        tz="America/Argentina/Cordoba",
    ),
    schedule=None,
    catchup=False,
    max_active_runs=1,
    default_args=default_args,
    tags=["proyecto-integrador", "vatsim", "grupo03"],
) as dag:

    extraer = BashOperator(
        task_id="extraer",
        bash_command=r"""
echo "=== DIAGNOSTICO DBT Y MINIO GRUPO03 ==="

echo ""
echo "1. Comprobando dbt:"
if command -v dbt >/dev/null 2>&1; then
    dbt --version
else
    echo "DBT NO ENCONTRADO"
fi

echo ""
echo "2. Comprobando librerias de Python:"
python - <<'PY'
import importlib.util

for paquete in ["duckdb", "dbt", "deltalake", "boto3"]:
    estado = (
        "DISPONIBLE"
        if importlib.util.find_spec(paquete)
        else "

## 3. Validar la sintaxis del DAG

Antes de desplegarlo, comprobamos que el código Python del DAG compila. Airflow no cargará un DAG con errores de sintaxis.

In [9]:
import ast

# Validar la sintaxis del código del DAG.
ast.parse(dag_code)

print("Sintaxis del DAG válida: no se encontraron errores.")

Sintaxis del DAG válida: no se encontraron errores.


## 4. Desplegar

Escribí el DAG a `dags/` y verificá en la UI de Airflow (http://localhost:8080) que aparezca. Activá el DAG y disparalo manualmente para la primera corrida.

```python
with open('/work/dags/pipeline_grupoX.py','w') as f:
    f.write(dag_code)
```

> **Backfill y reintentos** (contenido de la semana 11): con `catchup=True` Airflow ejecuta las corridas históricas pendientes; `retries` reintenta las tareas que fallan.

In [26]:

import ast
from pathlib import Path
from entorno import DAGS_DIR

assert "DIAGNOSTICO DBT Y MINIO GRUPO03" in dag_code, (
    "Primero ejecutá la celda 2."
)

ast.parse(dag_code)

ruta = Path(DAGS_DIR) / "pipeline_grupo03.py"
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(dag_code, encoding="utf-8")

print("DAG guardado en:", ruta)
print("Diagnóstico guardado:", True)



DAG guardado en: /home/jovyan/work/dags/pipeline_grupo03.py
Diagnóstico guardado: True


## Entrega 3 (E3)

Lo que se evalúa: pipeline orquestado con Airflow (scheduling, reintentos, backfill) e integración continua sobre el repositorio. Enunciado y rúbrica en la semana 10.

### CELDA TEMPORAL

In [27]:

from pathlib import Path
import yaml

repo = Path("/home/jovyan/work/INGDD-UCA2026")
compose = repo / "docker-compose.yml"

print("=== CONFIGURACION DOCKER ===")
print("Existe docker-compose:", compose.is_file())

if compose.is_file():
    configuracion = yaml.safe_load(
        compose.read_text(encoding="utf-8")
    )

    servicios = configuracion.get("services", {})

    for nombre, servicio in servicios.items():
        print(f"\nSERVICIO: {nombre}")
        print("Imagen:", servicio.get("image", "No definida"))
        print("Build:", servicio.get("build", "No definido"))
        print("Volumenes:", servicio.get("volumes", []))

print("\n=== ARCHIVOS DOCKER ===")

carpeta_docker = repo / "docker"

if carpeta_docker.is_dir():
    for archivo in sorted(carpeta_docker.rglob("*")):
        if archivo.is_file():
            print("-", archivo.relative_to(repo))


=== CONFIGURACION DOCKER ===
Existe docker-compose: True

SERVICIO: minio
Imagen: minio/minio:RELEASE.2024-06-13T22-53-53Z
Build: No definido
Volumenes: ['minio-data:/data']

SERVICIO: minio-setup
Imagen: minio/mc:RELEASE.2024-06-12T14-34-03Z
Build: No definido
Volumenes: []

SERVICIO: postgres
Imagen: postgres:16
Build: No definido
Volumenes: ['postgres-data:/var/lib/postgresql/data', './scripts/init-db.sql:/docker-entrypoint-initdb.d/init-db.sql:ro']

SERVICIO: airflow
Imagen: No definida
Build: {'context': './docker/airflow', 'dockerfile': 'Dockerfile'}
Volumenes: ['./dags:/opt/airflow/dags', './scripts:/opt/airflow/scripts', './dbt:/opt/airflow/dbt', 'airflow-db-data:/opt/airflow/logs']

SERVICIO: metabase
Imagen: metabase/metabase:v0.49.13
Build: No definido
Volumenes: ['metabase-data:/metabase-data']

=== ARCHIVOS DOCKER ===
- docker/airflow/.ipynb_checkpoints/Dockerfile-checkpoint
- docker/airflow/.ipynb_checkpoints/requirements-checkpoint.txt
- docker/airflow/Dockerfile
- docke

In [1]:
%pip install deltalake==1.6.6 pyarrow==26.0.0

Note: you may need to restart the kernel to use updated packages.
